# Food Insecurity and Meal Costs Analysis

This notebook investigates whether U.S. counties with lower food access have higher average meal costs. Analysis would include a permuation test and bootstrap confidence intervals.

## Imports

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style = 'whitegrid')

## Data Loading and Cleaning portion

### Feeding America Map the Meal Gap (2025 Report - 2023 Data)
County-level food insecurity rates and meal costs. Filtered ot 2023 data only.
Source: https://www.feedingamerica.org/research/map-the-meal-gap

In [6]:
# Load Feeding America data
meal_gap = pd.read_excel('data/map_the_meal_gap.xlsx', sheet_name = 'County', header = 0)
# Filter 2023 items only
meal_gap = meal_gap[meal_gap['Year'] == 2023]
# Keeping it small and relevant - focused columns:
meal_gap = meal_gap[['FIPS', 'State', 'County, State', 'Year', 'Overall Food Insecurity Rate', 'Cost Per Meal']]

# renaming - some are too long
meal_gap = meal_gap.rename(columns = {
    'County, State': 'county_name',
    'Overall Food Insecurity Rate': 'food_insecurity_rate',
    'Cost per Meal': 'cost_per_meal'
})

print(meal_gap.shape)
print(meal_gap.head())
print(meal_gap.dtypes)

(3144, 6)
    FIPS State              county_name  Year  food_insecurity_rate  \
4   1001    AL  Autauga County, Alabama  2023                 0.151   
9   1003    AL  Baldwin County, Alabama  2023                 0.148   
14  1005    AL  Barbour County, Alabama  2023                 0.197   
19  1007    AL     Bibb County, Alabama  2023                 0.193   
24  1009    AL   Blount County, Alabama  2023                 0.173   

    Cost Per Meal  
4            3.64  
9            4.01  
14           3.45  
19           3.36  
24           3.36  
FIPS                      int64
State                       str
county_name                 str
Year                      int64
food_insecurity_rate    float64
Cost Per Meal           float64
dtype: object


### USDA Food Enviornment Atlas (2025)
County-level food access inidcators. PCT_LACCESS_POP19 (percentage of total population with low food access) and PTC_LACCESS_LOW19 (percentage of low-income population with low food access) from 2019; 2019 is the most recent year that is avaliable.

Source: https://www.ers.usda.gov/data-products/food-environment-atlas

In [9]:
# Load Food Environment Atlas - row 2 is headers
atlas = pd.read_excel('data/2025_Food_Atlas.xlsx', sheet_name = 'ACCESS', header = 1)

# Necessary Columns:
atlas = atlas[['FIPS', 'State', 'County', 'PCT_LACCESS_POP19', 'PCT_LACCESS_LOWI19']]

# Renaming
atlas = atlas.rename(columns = {
    'PCT_LACCESS_POP19': 'pct_low_access',
    'PCT_LACCESS_LOWI19': 'pct_low_access_lowincome'
})

# Drop missing values (-9999)
atlas = atlas[atlas['pct_low_access'] != -9999]
atlas = atlas[atlas['pct_low_access_lowincome'] != -9999]

print(atlas.shape)
print(atlas.head())
print(atlas.dtypes)



(3142, 5)
   FIPS State   County  pct_low_access  pct_low_access_lowincome
0  1001    AL  Autauga       33.906700                 13.020998
1  1003    AL  Baldwin       25.122238                  7.936779
2  1005    AL  Barbour       20.520679                 10.433171
3  1007    AL     Bibb        1.593457                  0.445866
4  1009    AL   Blount        6.807624                  2.512206
FIPS                          int64
State                           str
County                          str
pct_low_access              float64
pct_low_access_lowincome    float64
dtype: object


### Mergining the Datasets
FIPS county code - join key

In [10]:
# FIPS same in both dataframe before the merge
meal_gap['FIPS'] = meal_gap['FIPS'].astype(str).str.zfill(5)
atlas['FIPS'] = atlas['FIPS'].astype(str).str.zfill(5)

# merge
df = meal_gap.merge(atlas, on = 'FIPS', how = 'inner')
# drop the duplicate state column
df = df.drop(columns = ['State_y'])
df = df.rename(columns = {'State_x': 'State'})

print(df.shape)
print(df.head())
print(df.isnull().sum())

(3133, 9)
    FIPS State              county_name  Year  food_insecurity_rate  \
0  01001    AL  Autauga County, Alabama  2023                 0.151   
1  01003    AL  Baldwin County, Alabama  2023                 0.148   
2  01005    AL  Barbour County, Alabama  2023                 0.197   
3  01007    AL     Bibb County, Alabama  2023                 0.193   
4  01009    AL   Blount County, Alabama  2023                 0.173   

   Cost Per Meal   County  pct_low_access  pct_low_access_lowincome  
0           3.64  Autauga       33.906700                 13.020998  
1           4.01  Baldwin       25.122238                  7.936779  
2           3.45  Barbour       20.520679                 10.433171  
3           3.36     Bibb        1.593457                  0.445866  
4           3.36   Blount        6.807624                  2.512206  
FIPS                        0
State                       0
county_name                 0
Year                        0
food_insecurity_rate   